## Shannon invariants: A scalable approach to information decomposition
Aaron J. Gutknecht, Fernando E. Rosas, David A. Ehrlich, Abdullah Makkeh, Pedro A. M. Mediano, Michael Wibral
Supplementary Code -  Script 1/5
### Training of the deep convolutional autoencoder for face images and computation of Shannon invariant summary structures (Figure 3)

Note: Running this script requires the "Labelled Faces in the Wild (LFW) dataset to be placed in the /data/ subfolder. The files can be obtained from https://www.kaggle.com/datasets/jessicali9530/lfw-dataset/data

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import time

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
cm = 1/2.54  # centimeters to inches

import numpy as np

import nninfo
from nninfo.plot import format_figure_broken_axis, plot_mean_and_interval

### Train network

In [ ]:
# Set experiment id
bottleneck_size = 128
experiment_id = f"lfw_autoencoder_b{bottleneck_size}"

#### Set up parameters and train first network

In [ ]:
# Note that we do not set initial seeds manually here, but save all seeds to the
# checkpoints files during training for later reproducibility. Rerunning this script
# will produce slightly different figures due to the randomness of network
# initialization etc.

layer_infos = [
    # Input layer
    nninfo.LayerInfo(connection_layer='input', activation_function='input'), # 32x32x3

    # Encoder
    nninfo.LayerInfo(connection_layer='conv2d', connection_layer_kwargs={'in_channels': 3, 'out_channels': 8, 'kernel_size': 3, 'stride': 1, 'padding': 1}, activation_function='tanh'),
    nninfo.LayerInfo(connection_layer='maxpool2d', connection_layer_kwargs={'kernel_size': 2, 'stride': 2}, activation_function='hardtanh'),
    nninfo.LayerInfo(connection_layer='conv2d', connection_layer_kwargs={'in_channels': 8, 'out_channels': 16, 'kernel_size': 3, 'stride': 1, 'padding': 1}, activation_function='tanh'),
    nninfo.LayerInfo(connection_layer='maxpool2d', connection_layer_kwargs={'kernel_size': 2, 'stride': 2}, activation_function='hardtanh'),
    nninfo.LayerInfo(connection_layer='conv2d', connection_layer_kwargs={'in_channels': 16, 'out_channels': 32, 'kernel_size': 3, 'stride': 1, 'padding': 1}, activation_function='tanh'),
    nninfo.LayerInfo(connection_layer='maxpool2d', connection_layer_kwargs={'kernel_size': 2, 'stride': 2}, activation_function='hardtanh'),
    nninfo.LayerInfo(connection_layer='flatten', activation_function=None),
    nninfo.LayerInfo(connection_layer='linear', connection_layer_kwargs={'in_features': 4*4*32, 'out_features': bottleneck_size}, activation_function='tanh'),

    # Decoder
    nninfo.LayerInfo(connection_layer='linear', connection_layer_kwargs={'in_features': bottleneck_size, 'out_features': 4*4*32}, activation_function='tanh'),
    nninfo.LayerInfo(connection_layer='reshape', connection_layer_kwargs={'shape': (-1, 32, 4, 4)}, activation_function=None),
    nninfo.LayerInfo(connection_layer='conv2d_transpose', connection_layer_kwargs={'in_channels': 32, 'out_channels': 16, 'kernel_size': 3, 'stride': 2, 'padding': 1, 'output_padding': 1}, activation_function='tanh'),
    nninfo.LayerInfo(connection_layer='conv2d_transpose', connection_layer_kwargs={'in_channels': 16, 'out_channels': 8, 'kernel_size': 3, 'stride': 2, 'padding': 1, 'output_padding': 1}, activation_function='tanh'),
    
    # Output layer
    nninfo.LayerInfo(connection_layer='conv2d_transpose', connection_layer_kwargs={'in_channels': 8, 'out_channels': 3, 'kernel_size': 3, 'stride': 2, 'padding': 1, 'output_padding': 1}, activation_function='sigmoid'),
]

# Set weight initialization
initializer_name = 'xavier'

# Create network instance
network = nninfo.NeuralNetwork(layer_infos=layer_infos, init_str=initializer_name)

# Set task instance
task = nninfo.TaskManager('face_2d_autoencoder_augmented')

# Split dataset into train, test, and validation sets
task['full_set'].train_test_val_sequential_split(100_000, 32_330, 0)

# Create quantizer list with stochastic quantization. The input layer is not quantized.
quantizer = [None, 
            {'levels': 8, 'rounding_point': 'stochastic'}, None,
            {'levels': 8, 'rounding_point': 'stochastic'}, None,
            {'levels': 8, 'rounding_point': 'stochastic'}, None,
            None,
            {'levels': 8, 'rounding_point': 'stochastic'}, {'levels': 8, 'rounding_point': 'stochastic'},
            None,
            {'levels': 8, 'rounding_point': 'stochastic'}, {'levels': 8, 'rounding_point': 'stochastic'},
            None]
#Set up trainer
trainer = nninfo.Trainer(dataset_name='full_set/train',
                        optim_str='Adam',
                        loss_str='MSELoss',
                        lr=0.001,
                        shuffle=True,
                        batch_size=32,
                        quantizer=quantizer,
                        )

# Set up tester
tester = nninfo.Tester(dataset_name='full_set/test')

# Set up schedule
schedule = nninfo.Schedule()
# Save training state for 50 logarithmically spaced checkpoints
schedule.create_log_spaced_chapters(1_000, 50, start_exp=-2, fractional=True)

# Combine components into experiment
experiment = nninfo.Experiment(experiment_id=experiment_id,
                        network=network,
                        task=task,
                        trainer=trainer,
                        tester=tester,
                        schedule=schedule)

# Run training for 10^3 epochs
experiment.run_following_schedule(use_cuda=False, compute_test_loss=False)

In [ ]:
# print python working directory

from pathlib import Path
print(f"Python working directory: {__file__}")

#### Rerun training with different random weight initializations

In [ ]:
# Set up experiment
experiment = nninfo.Experiment.load(experiment_id)

# Compute 19 more training runs with different random weight initializations.
experiment.rerun(19)

### Evaluate network performance
#### Compute loss and accuracy

In [ ]:
experiment = nninfo.Experiment.load(experiment_id)
performance_measurement = nninfo.analysis.PerformanceMeasurement(experiment, ['full_set/train', 'full_set/test'], quantizer_params=quantizer)

performance_measurement.perform_measurements(run_ids='all', chapter_ids='all', exists_ok=True)

#### Plot loss and accuracy

In [ ]:
# Load performance file
experiment = nninfo.Experiment.load(experiment_id)
performance_measurement = nninfo.analysis.PerformanceMeasurement.load(experiment, "performance")

fig, ax = plt.subplots(figsize=(4*cm, 4*cm), dpi=150)
ax.set_ylim(0, 1)
twinax = ax.twinx()

# Plot accuracy
nninfo.plot.plot_loss_accuracy(performance_measurement.results, ax, twinax)

ax.legend(ncol=1, bbox_to_anchor=(1.5, 0.5), loc='center left');

In [ ]:
# Save result
plt.savefig(f"experiments/exp_{experiment_id}/plots/performance.pdf", bbox_inches='tight')

In [ ]:
# Visually inspect reconstructions

experiment = nninfo.Experiment.load(experiment_id)
output_activations = experiment.tester._get_output_activations('full_set/train', quantizer_params=quantizer)

y_pred, y = next(output_activations)

fig, axs = plt.subplots(2, 5, figsize=(2*5*cm, 2*2*cm))
for i in range(5):
    axs[0, i].imshow(y[i].reshape(3, 32, 32).swapaxes(0, 2).swapaxes(0, 1))
    axs[0, i].axis('off')
    axs[1, i].imshow(y_pred[i].reshape(3, 32, 32).swapaxes(0, 2).swapaxes(0, 1))
    axs[1, i].axis('off')

### Compute Shannon invariant summary structures on $L_3$, $L_4$ and $L_5$

In [ ]:
# Load experiment
experiment = nninfo.Experiment.load(experiment_id)

layer_shapes = {2: (8, 32, 32), 3: (8, 16, 16), 5: (16, 8, 8), 7: (32, 4, 4), 9: (128,), 11: (32, 4, 4), 12: (16, 8, 8), 13: (8, 16, 16), 14: (3, 32, 32)}
    
for layer in [3, 5, 7, 9, 11, 12, 13]:
    sources_shape = layer_shapes[layer]

    target = []
    if len(sources_shape) == 1:
        sources = [[nninfo.NeuronID(f'L{layer}', (i+1,))] for i in range(sources_shape[0])]
    else:
        sources = [[nninfo.NeuronID(f'L{layer}', tuple(i+1 for i in idx))] for idx in np.ndindex(sources_shape)]

    
    measurement = nninfo.analysis.PIDMeasurement(experiment=experiment,
                                                measurement_id=f'summary_measurement_L{layer}',
                                                dataset_name='full_set/train',
                                                quantizer_params=quantizer,
                                                pid_definition='deg_red',
                                                pid_kwargs={'entropy_only': True},
                                                binning_kwargs={'binning_method' : 'none'},
                                                target_id_list=target,
                                                source_id_lists=sources)
    
    start = time.perf_counter()
    measurement.perform_measurements(run_ids='all', chapter_ids='all', exists_ok=True)
    stop = time.perf_counter()

#### Compute degree of redundancy and vulnerability from measurements and plot

In [ ]:
def compute_degred(df):
    """ Compute average degree reduction for each column containing the mutual information with an individual source.
    """

    df['avg_deg_red'] = df.filter(regex='^source_MI[0-9]*$', axis=1).sum(axis=1) / df['total_MI']

    return df

def compute_vuln(df):
    """ Compute vulnerability for each column containing the mutual information with all but one source.
    """

    num_sources = len(df.filter(regex='^allbut_source_MI[0-9]*$', axis=1).columns)
    df['vuln'] = (num_sources*df['total_MI'] - df.filter(regex='^allbut_source_MI[0-9]*$', axis=1).sum(axis=1)) / df['total_MI']

    return df

In [ ]:
# Plot degree of redundancy for each layer

fig, ax= plt.subplots(figsize=(5*cm, 4*cm), dpi=150)
format_figure_broken_axis(ax)

experiment = nninfo.experiment.Experiment.load(experiment_id)

# Use colors from blue to red in the order of the layers
colors = [plt.cm.get_cmap('coolwarm', 7)(i) for i in range(7)]
colors[3] = 'tab:green'

for layer, color in zip([3, 5, 7, 9, 11, 12, 13], colors):
    summary_measurement = nninfo.analysis.PIDMeasurement.load(experiment=experiment, measurement_id=f'summary_measurement_L{layer}')
    summary_df = compute_degred(summary_measurement.results)
    plot_mean_and_interval(summary_df, ax, x='epoch_id', y='avg_deg_red', label=f'Layer {layer}', color=color)

ax.set_xlabel('Epoch')
ax.set_ylabel('Deg. of Redundancy')
ax.set_xlim(0, 1000)

ax.legend(bbox_to_anchor=(1, .6), loc='upper left')

# Save results
plt.savefig(f"../experiments/exp_{experiment_id}/plots/degree_of_redundancy.pdf", bbox_inches='tight')



In [ ]:
# Plot degree of vulnerability for each layer

fig, ax= plt.subplots(figsize=(5*cm, 4*cm), dpi=150)
format_figure_broken_axis(ax)

experiment = nninfo.experiment.Experiment.load(experiment_id)

# Use colors from blue to red in the order of the layers
colors = [plt.cm.get_cmap('coolwarm', 7)(i) for i in range(7)]
colors[3] = 'tab:green'

for layer, color in zip([3, 5, 7, 9, 11, 12, 13], colors):
    summary_measurement = nninfo.analysis.PIDMeasurement.load(experiment=experiment, measurement_id=f'summary_measurement_L{layer}')
    summary_df = compute_vuln(summary_measurement.results)
    plot_mean_and_interval(summary_df, ax, x='epoch_id', y='vuln', label=f'Layer {layer}', color=color)

ax.set_xlabel('Epoch')
ax.set_ylabel('Deg. of Vulnerability')
ax.set_xlim(0, 1000)

ax.legend(bbox_to_anchor=(1, .6), loc='upper left')

# Save results
plt.savefig(f"../experiments/exp_{experiment_id}/plots/degree_of_vulnerability.pdf", bbox_inches='tight')